# Sinhala F5-TTS fine-tuning on Kaggle

This notebook fine-tunes F5-TTS on the combined OpenSLR + PathNirvana corpus while preserving Sinhala Unicode text. It works in Kaggle or Google Colab. It first tries the Sinhala checkpoint; if gated approval is pending, it automatically transfers the public F5 base weights and trains a Sinhala Unicode text adapter.

In [ ]:
!pip -q install f5-tts==1.1.21 tensorboard
import os, sys, json, csv, tarfile, shutil, subprocess
from pathlib import Path
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
import os
from pathlib import Path
from huggingface_hub import hf_hub_download
from huggingface_hub.errors import GatedRepoError
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = os.environ.get('HF_TOKEN')
ROOT = Path('/kaggle/working/sinhala_f5' if Path('/kaggle/working').exists() else '/content/sinhala_f5')
ROOT.mkdir(parents=True, exist_ok=True)
ckpt_dir = ROOT / 'checkpoint'
ckpt_dir.mkdir(exist_ok=True)
MODEL_PATH = VOCAB_PATH = PUBLIC_MODEL_PATH = None
USE_PUBLIC_BASE = False
for f in ['ckpts/f5_TTS/model_230000_reduced.pt', 'ckpts/f5_TTS/vocab.txt']:
    try:
        if not HF_TOKEN: raise RuntimeError('no token')
        p = Path(hf_hub_download('tharindumihi/tts-si-F5-TTS', f, token=HF_TOKEN, local_dir=str(ckpt_dir)))
    except (GatedRepoError, RuntimeError):
        USE_PUBLIC_BASE = True
        break
    if p.suffix == '.pt': MODEL_PATH = p
    else: VOCAB_PATH = p
    print('Downloaded:', p)
if USE_PUBLIC_BASE:
    print('Sinhala checkpoint is gated/pending; using public F5 base with Sinhala text-embedding transfer.')
    PUBLIC_MODEL_PATH = Path(hf_hub_download('SWivid/F5-TTS', 'F5TTS_v1_Base/model_1250000.safetensors', local_dir=str(ckpt_dir)))
else:
    if MODEL_PATH is None or VOCAB_PATH is None: raise RuntimeError('Checkpoint download did not return both model and vocab files.')
print('MODEL_PATH:', MODEL_PATH)
print('PUBLIC_MODEL_PATH:', PUBLIC_MODEL_PATH)
print('VOCAB_PATH:', VOCAB_PATH)

In [ ]:
# Download the public combined OpenSLR + PathNirvana archive (~1.72 GB).
archive = ROOT / 'data.tar.gz'
if not archive.exists() or archive.stat().st_size < 1_700_000_000:
    url = 'https://huggingface.co/datasets/keshan/multispeaker-tts-sinhala/resolve/main/data.tar.gz'
    subprocess.run(['wget', '-c', '--show-progress', url, '-O', str(archive)], check=True)
index = ROOT / 'file_index.tsv'
if not index.exists():
    hf_hub_download('keshan/multispeaker-tts-sinhala', 'file_index.tsv', repo_type='dataset', local_dir=str(ROOT))
audio_root = ROOT / 'audio'
if not audio_root.exists():
    audio_root.mkdir()
    with tarfile.open(archive, 'r:gz') as t: t.extractall(audio_root)
print('archive', archive.stat().st_size, 'audio root', audio_root)

In [ ]:
# Create absolute-path F5 manifests and a Sinhala character vocabulary.
items = []
with (ROOT/'file_index.tsv').open(encoding='utf-8') as h:
    for r in csv.DictReader(h, delimiter='\t'):
        hits = list(audio_root.rglob(r['file_path']))
        if hits and r['sentence'].strip():
            items.append((str(hits[0].resolve()), r['sentence'].strip()))
print('usable items:', len(items))
train, val = [], []
for audio, text in items:
    name = Path(audio).name
    (val if name.split('_')[-1][:1] in {'0','1'} else train).append((audio,text))
manifest = ROOT/'manifest'; manifest.mkdir(exist_ok=True)
for name, rows in [('train.csv',train),('val.csv',val)]:
    with (manifest/name).open('w',encoding='utf-8',newline='') as h:
        w=csv.writer(h,delimiter='|',lineterminator='\n'); w.writerow(['audio_file','text']); w.writerows(rows)
chars=sorted({c for _,t in items for c in t if c!=' '})
(manifest/'vocab.txt').write_text(' \n'+'\n'.join(chars)+'\n', encoding='utf-8')
print('train',len(train),'val',len(val),'vocab',len(chars)+1)

In [ ]:
# Convert audio paths to the official F5 Arrow format.
from f5_tts.model.utils import get_tokenizer
from f5_tts.model.dataset import CustomDataset
arrow = ROOT/'arrow'; arrow.mkdir(exist_ok=True)
subprocess.run([sys.executable, '-m', 'f5_tts.train.datasets.prepare_csv_wavs', str(manifest/'train.csv'), str(arrow), '--pretrain', '--workers', '4'], check=True)
# The official pretrain helper generates a generic vocabulary; use the Sinhala one.
shutil.copy2(manifest/'vocab.txt', arrow/'vocab.txt')
print('prepared', arrow/'raw.arrow')

In [ ]:
# Fine-tune: conservative LR, frame batching, gradient accumulation, 8-bit optimizer.
from datasets import Dataset
from f5_tts.model import CFM, DiT, Trainer
TRAIN_VOCAB = VOCAB_PATH if VOCAB_PATH is not None else manifest/'vocab.txt'
vocab_map, vocab_size = get_tokenizer(str(TRAIN_VOCAB), 'custom')
mel = dict(target_sample_rate=24000,n_mel_channels=100,hop_length=256,win_length=1024,n_fft=1024,mel_spec_type='vocos')
model = CFM(transformer=DiT(dim=1024,depth=22,heads=16,ff_mult=2,text_dim=512,conv_layers=4,text_num_embeds=vocab_size,mel_dim=100,checkpoint_activations=True), mel_spec_kwargs=mel, vocab_char_map=vocab_map)
run_dir = ROOT/'finetuned'; run_dir.mkdir(exist_ok=True)
if not USE_PUBLIC_BASE:
    shutil.copy2(MODEL_PATH, run_dir/'pretrained_model_230000_reduced.pt')
trainer = Trainer(model, epochs=5, learning_rate=1e-5, num_warmup_updates=500, save_per_updates=500, keep_last_n_checkpoints=2, checkpoint_path=str(run_dir), batch_size_per_gpu=6000, batch_size_type='frame', max_samples=4, grad_accumulation_steps=4, max_grad_norm=1.0, logger=None, last_per_updates=250, bnb_optimizer=True, mel_spec_type='vocos')
if USE_PUBLIC_BASE:
    from safetensors.torch import load_file
    base = load_file(str(PUBLIC_MODEL_PATH), device='cpu')
    target = trainer.accelerator.unwrap_model(trainer.model).state_dict()
    compatible = {}
    for key, value in base.items():
        key = key.replace('ema_model.', '', 1) if key.startswith('ema_model.') else key
        if key in target and target[key].shape == value.shape: compatible[key] = value
    trainer.accelerator.unwrap_model(trainer.model).load_state_dict(compatible, strict=False)
    if trainer.is_main: trainer.ema_model.load_state_dict(compatible, strict=False)
    print('Transferred compatible public F5 parameters:', len(compatible), '/', len(target))
update = 0 if USE_PUBLIC_BASE else trainer.load_checkpoint()
ds = Dataset.from_file(str(arrow/'raw.arrow'))
durations = json.loads((arrow/'duration.json').read_text())['duration']
train_ds = CustomDataset(ds, durations=durations, **mel)
trainer.train(train_ds, num_workers=2, resumable_with_seed=666)
print('finished at update', update, 'checkpoints:', list(run_dir.glob('model_*.pt')))

In [ ]:
# Generate a sample using the final checkpoint and a PathNirvana reference clip.
from f5_tts.infer.utils_infer import load_model, load_vocoder, infer_process
from f5_tts.model import DiT
final_ckpt = sorted(run_dir.glob('model_*.pt'), key=lambda p: p.stat().st_mtime)[-1]
ref_audio, ref_text = items[0]
infer_model = load_model(DiT, dict(dim=1024,depth=22,heads=16,ff_mult=2,text_dim=512,conv_layers=4), str(final_ckpt), vocab_file=str(TRAIN_VOCAB), device='cuda')
vocoder = load_vocoder(vocoder_name='vocos', is_local=False)
audio, sr, _ = infer_process(ref_audio, ref_text, 'අද අපි ස්වභාවික සිංහල හඬක් පරීක්ෂා කරමු.', infer_model, vocoder, nfe_step=32, cfg_strength=2.0, sway_sampling_coef=-1.0, device='cuda')
import soundfile as sf
sample_path = ROOT/'sinhala_f5_sample.wav'
sf.write(sample_path, audio, sr)
from IPython.display import Audio, display
display(Audio(str(sample_path)))